In [2]:
import pandas as pd
import numpy as np

# This notebook stands on its own, so we reload everything here
results = pd.read_csv("../data/Raw/ironman703agadir2025-results.csv")
acts    = pd.read_csv("../data/Raw/activities.csv")

print("results:", results.shape)
print("activities:", acts.shape)

results: (523, 34)
activities: (266, 103)


In [3]:
def time_to_seconds(t):
    if pd.isna(t) or not isinstance(t, str):
        return np.nan
    parts = [int(p) for p in t.strip().split(":")]
    if len(parts) == 3:
        h, m, s = parts
        return h*3600 + m*60 + s
    elif len(parts) == 2:
        m, s = parts
        return m*60 + s
    return np.nan

time_cols = ['Swim Time', 'Bike Time', 'Run Time',
             'Transition 1 Time', 'Transition 2 Time', 'Overall Time']
for col in time_cols:
    results[col + ' (s)'] = results[col].apply(time_to_seconds)

results[['Name','Swim Time','Swim Time (s)']].head()

,Name,Swim Time,Swim Time (s)
0,Christophe Halleumieux,00:29:24,1764
1,Korbinian Rudzki,00:27:32,1652
2,Burkhard Schmidt,00:31:33,1893
3,Andreas Bund,00:33:13,1993
4,Alejandro Lopez Valenciano,00:28:26,1706


In [4]:
me       = results[results['Name'].str.contains('GHILANI', case=False, na=False)]
ag_ref   = results[results['Division'] == 'M18-24'].copy()
male_ref = results[results['Gender'] == 'Male'].copy()

print("Found rows for you:", len(me))     # should be 1
print("M18-24:", len(ag_ref), "| Males:", len(male_ref))

Found rows for you: 1
M18-24: 26 | Males: 459


In [5]:
def zscores_vs_group(me, group, label):
    print(f"\n===== Z-scores: You vs {label} (n={len(group)}) =====")
    for col in ['Swim Time (s)', 'Bike Time (s)', 'Run Time (s)']:
        my_time = me[col].values[0]
        avg     = group[col].mean()
        std     = group[col].std()
        z       = (my_time - avg) / std
        print(f"{col:16s} | gap: {my_time-avg:+6.0f}s | spread: {std:5.0f}s | z = {z:+.2f}")

zscores_vs_group(me, ag_ref,   "M18-24")
zscores_vs_group(me, male_ref, "All males")


===== Z-scores: You vs M18-24 (n=26) =====
Swim Time (s)    | gap:   +882s | spread:   542s | z = +1.63
Bike Time (s)    | gap:  +4205s | spread:  3054s | z = +1.38
Run Time (s)     | gap:  +4117s | spread:  2113s | z = +1.95

===== Z-scores: You vs All males (n=459) =====
Swim Time (s)    | gap:   +779s | spread:   775s | z = +1.01
Bike Time (s)    | gap:  +4235s | spread:  3425s | z = +1.24
Run Time (s)     | gap:  +3835s | spread:  2581s | z = +1.49


In [6]:
runs = acts[acts['Activity Type'] == 'Run'].copy()
runs['date'] = pd.to_datetime(runs['Activity Date'],
                              format='%b %d, %Y, %I:%M:%S %p',
                              errors='coerce')

# The file has DUPLICATE column names — inspect which distance/time cols exist
print([c for c in runs.columns if 'Distance' in c or 'Elapsed' in c or 'Moving' in c])

['Elapsed Time', 'Distance', 'Elapsed Time.1', 'Moving Time', 'Distance.1', 'Grade Adjusted Distance', 'Average Elapsed Speed', 'Dirt Distance', 'Newly Explored Distance', 'Newly Explored Dirt Distance', 'Downhill Distance']


In [7]:
# Look first — don't assume
print(runs[['Distance', 'Distance.1']].head())

    Distance  Distance.1
6      10.25     10249.7
9       5.01      5012.6
11      6.51      6513.8
16     12.00     12003.1
19      5.94      5945.4


In [8]:
runs['dist_km']     = runs['Distance.1'] / 1000      # <-- use the METRES column
runs['elapsed_s']   = runs['Elapsed Time']           # <-- the seconds column
runs['pace_min_km'] = (runs['elapsed_s'] / 60) / runs['dist_km']

# Sanity check: pace should be roughly 4–9 min/km
runs[['date','Activity Name','dist_km','elapsed_s','pace_min_km']].head(10)

,date,Activity Name,dist_km,elapsed_s,pace_min_km
6,2026-07-11 05:49:04,Morning Run,10.2497,3490,5.674963
9,2026-07-07 19:45:42,Evening Run,5.0126,2993,9.951589
11,2026-07-03 05:53:17,Morning Run,6.5138,2327,5.954026
16,2026-06-24 05:43:30,Morning Run,12.0031,4626,6.423341
19,2026-06-22 06:31:44,Morning Run,5.9454,2089,5.856068
20,2026-06-20 18:16:13,Push,0.0000,4020,inf
26,2026-06-16 14:02:23,Afternoon Run,3.3333,1200,6.000060
28,2026-06-15 07:03:00,Morning Run,8.0029,2656,5.531328
32,2026-06-10 05:09:30,Morning Run,1.7702,825,7.767484
34,2026-06-09 06:59:58,Morning recovery run,4.0075,1683,6.999376


In [9]:
race_date = pd.Timestamp('2025-10-26')

window = runs[(runs['date'] >= '2025-09-15') & (runs['date'] <= '2025-12-15')].copy()
window = window.sort_values('date')

# Drop the junk rows (inf/NaN pace) just for reading
window = window[np.isfinite(window['pace_min_km'])]

window[['date','Activity Name','dist_km','pace_min_km']]

,date,Activity Name,dist_km,pace_min_km
223,2025-09-21 08:02:39,Long run,19.4188,6.913404
222,2025-09-27 08:30:17,Morning Run,3.4791,5.451601
221,2025-09-28 21:32:16,Night Run,9.8926,5.888240
220,2025-10-01 08:00:30,Morning treadmill run,6.8160,4.401408
219,2025-10-03 08:55:22,3 weeks out 🔥,6.7196,5.493879
218,2025-10-05 06:14:17,Morning Run,2.9309,6.687366
217,2025-10-08 21:36:00,Night Run,7.6095,5.994700
216,2025-10-12 12:47:02,Afternoon Run,3.9283,5.651300
215,2025-10-12 19:52:22,Night Run,7.7237,6.844733
214,2025-10-17 08:04:42,Recovery run,3.7047,7.517478


In [10]:
#it looks like the the average run pace before the race was about (5:30 min/km) and the race pace is 7:45 min/km, which is a significant drop in performance.
# This IS probably due to various factors such as fatigue, injury, dehydration and nutrition. 


In [11]:
# Weekly running volume in the lead-up to the race
runs_2025 = runs[(runs['date'] >= '2025-08-01') & (runs['date'] <= '2025-10-26')].copy()

# Group by ISO week, sum the distance
weekly = (runs_2025
          .set_index('date')['dist_km']
          .resample('W')          # 'W' = calendar week
          .sum())

print(weekly.round(1))

date
2025-08-03    10.0
2025-08-10    31.8
2025-08-17     3.4
2025-08-24    18.3
2025-08-31    24.1
2025-09-07    23.8
2025-09-14    16.7
2025-09-21    19.4
2025-09-28    13.4
2025-10-05    16.5
2025-10-12    19.3
2025-10-19    11.1
Freq: W-SUN, Name: dist_km, dtype: float64


In [12]:
# WEEKLY MILEAGE FINDING:
# Run volume was LOW and INCONSISTENT (10-32 km/wk, big week-to-week swings),
# not a structured progressive build. No single steep pre-race spike; rather
# chronic under-structuring. Consistent with elevated injury risk when combined
# with physio-assessed glute weakness. Also implies run was under-trained in volume.

In [13]:
# Save your per-discipline z-scores for later phases (ROI, dashboard)
import os
os.makedirs("../data/processed", exist_ok=True)

diagnosis = pd.DataFrame({
    'discipline': ['Swim','Bike','Run'],
    'my_time_s':  [me['Swim Time (s)'].values[0],
                   me['Bike Time (s)'].values[0],
                   me['Run Time (s)'].values[0]],
    'z_ag':   [1.63, 1.38, 1.95],      # from your z-score cell
    'z_male': [1.01, 1.24, 1.49],
})
diagnosis.to_csv("../data/processed/diagnosis.csv", index=False)
diagnosis

,discipline,my_time_s,z_ag,z_male
0,Swim,2763,1.63,1.01
1,Bike,13291,1.38,1.24
2,Run,10011,1.95,1.49


In [ ]:
# results already loaded in cell 0; no external prep module needed
print([c for c in results.columns if 'corrected' in c])
results = load_results()
print([c for c in results.columns if 'corrected' in c])

ModuleNotFoundError: No module named 'prep'